# 02 — The DuckDB warehouse

**Purpose.** Load the validated dataset into an analytical store and define the SQL
views that the dashboard and the later analysis features read.

**Why a warehouse at all**, when the data is 64,000 rows and fits comfortably in a
DataFrame? Three reasons, and none of them is size:

1. **One definition per metric.** "Visit rate" is defined once, in SQL, rather than
   recomputed in every notebook and every dashboard callback where it can silently
   drift.
2. **The dashboard needs a query interface.** Feature 11 slices by arm and segment
   interactively. Pushing that into SQL keeps the Streamlit layer thin.
3. **It is the honest shape of the problem.** In production this data lives in a
   warehouse, not a CSV. Modelling it that way makes the project's structure
   transferable rather than notebook-specific.

The store is a *derived artifact* — rebuilt from the processed parquet, never
edited in place, and not version-controlled.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.db.warehouse import build, query, table, sql_files

pd.set_option("display.width", 120)

build()
print("Views defined in:", [p.name for p in sql_files()])

Views defined in: ['01_arm_metrics.sql', '02_arm_lift.sql', '03_funnel.sql', '04_customer_dimensions.sql', '05_segment_metrics.sql']


## 1. What is in the store

One table and five views. The numeric prefixes on the SQL files encode the
dependency order — `v_arm_lift` reads `v_arm_metrics`, and `v_segment_metrics` reads
`v_customer_dimensions`.

In [2]:
query("""
    SELECT table_name, table_type
    FROM information_schema.tables
    ORDER BY table_type, table_name
""")

,table_name,table_type
0,customers,BASE TABLE
1,v_arm_lift,VIEW
2,v_arm_metrics,VIEW
3,v_customer_dimensions,VIEW
4,v_funnel,VIEW
5,v_segment_metrics,VIEW


### A note on `customer_id`

The base table gets a surrogate row key so the unpivoted dimension view can be
joined back to it. It identifies a **row, not a person** — the dataset has no
customer identifier, and the 6,562 duplicated rows are retained deliberately
(Feature 1). Nothing in this project treats `customer_id` as a person.

## 2. Headline metrics by arm

In [3]:
table("v_arm_metrics").style.format({
    "customers": "{:,}",
    "visitors": "{:,}",
    "converters": "{:,}",
    "visit_rate": "{:.2%}",
    "conversion_rate": "{:.3%}",
    "mean_spend": "${:.3f}",
    "total_spend": "${:,.0f}",
    "mean_spend_per_converter": "${:.2f}",
    "spend_stddev": "${:.2f}",
}).hide(axis="index")

arm,customers,visitors,converters,visit_rate,conversion_rate,mean_spend,total_spend,mean_spend_per_converter,spend_stddev
No E-Mail,"21,306","2,262.0",122.0,10.62%,0.573%,$0.653,"$13,908",$114.00,$11.59
Mens E-Mail,"21,307","3,894.0",267.0,18.28%,1.253%,$1.423,"$30,312",$113.53,$17.75
Womens E-Mail,"21,387","3,238.0",189.0,15.14%,0.884%,$1.077,"$23,038",$121.89,$15.12


Note `spend_stddev` against `mean_spend`: a standard deviation around **$15 on a
mean near $1**. That ratio is the single biggest obstacle in this project, and it is
carried in the view precisely so it stays visible rather than emerging as a surprise
in the power analysis.

## 3. Lift over control

The three outcomes are unpivoted into rows, so one view serves all of them.

In [4]:
table("v_arm_lift").style.format({
    "treatment_value": "{:.4f}",
    "control_value": "{:.4f}",
    "absolute_lift": "{:+.4f}",
    "relative_lift": "{:+.1%}",
}).hide(axis="index")

arm,outcome,treatment_value,control_value,absolute_lift,relative_lift
Mens E-Mail,conversion_rate,0.0125,0.0057,+0.0068,+118.8%
Mens E-Mail,mean_spend,1.4226,0.6528,+0.7698,+117.9%
Mens E-Mail,visit_rate,0.1828,0.1062,+0.0766,+72.1%
Womens E-Mail,conversion_rate,0.0088,0.0057,+0.0031,+54.3%
Womens E-Mail,mean_spend,1.0772,0.6528,+0.4244,+65.0%
Womens E-Mail,visit_rate,0.1514,0.1062,+0.0452,+42.6%


Large relative lifts — Mens E-Mail roughly doubles conversion and spend.

**These numbers carry no uncertainty.** A relative lift of +117.9% on spend is a
ratio of two noisy means, one of which has a standard deviation 14x its own value.
The view reports the point estimate and stops there; standard errors, confidence
intervals and significance are Feature 4's job. That separation is deliberate — SQL
makes it easy to compute a difference and awkward to compute the uncertainty around
it, which is a good reason not to let inference live here.

## 4. Where in the funnel does the effect sit?

A lift in conversion can come from two places: more people arriving, or the people
who arrive being likelier to buy. The funnel view separates them.

In [5]:
table("v_funnel").style.format({
    "assigned": "{:,}",
    "visited": "{:,}",
    "converted": "{:,}",
    "visit_rate": "{:.2%}",
    "conversion_rate": "{:.3%}",
    "conversion_rate_given_visit": "{:.2%}",
    "spend_per_converter": "${:.2f}",
}).hide(axis="index")

arm,assigned,visited,converted,visit_rate,conversion_rate,conversion_rate_given_visit,spend_per_converter
No E-Mail,"21,306","2,262.0",122.0,10.62%,0.573%,5.39%,$114.00
Mens E-Mail,"21,307","3,894.0",267.0,18.28%,1.253%,6.86%,$113.53
Womens E-Mail,"21,387","3,238.0",189.0,15.14%,0.884%,5.84%,$121.89


Both channels appear to move: Mens E-Mail raises the visit rate from 10.6% to 18.3%,
*and* visitors convert at 6.9% versus 5.4% in control.

**But the second of those is not a causal comparison.** `conversion_rate_given_visit`
conditions on visiting — and visiting is itself affected by the treatment. The email
pulls in extra visitors, and those marginal visitors need not resemble the people who
would have visited anyway, so the two conditional rates are computed over
non-comparable populations. This is collider bias, and it is exactly the kind of
metric that looks like a finding in a dashboard and is not one.

It is kept as a *diagnostic* for where the effect plausibly sits, labelled as
non-causal in the view definition and wherever it is displayed. Notably,
`spend_per_converter` is roughly flat across arms (\$114 / \$114 / \$122), which is
consistent with the emails changing *how many* people buy rather than how much each
buyer spends.

## 5. Slicing by customer attributes

`v_customer_dimensions` reshapes seven pre-treatment attributes from one column each
into `(customer_id, dimension, level)` rows. That turns "compute metrics for every
way of slicing customers" from seven near-identical queries into one GROUP BY with
an extra key.

In [6]:
query("""
    SELECT dimension, count(DISTINCT level) AS levels, count(*) AS rows
    FROM v_customer_dimensions
    GROUP BY dimension
    ORDER BY levels DESC, dimension
""")

,dimension,levels,rows
0,Prior spend band,7,64000
1,Recency,4,64000
2,Location,3,64000
3,Purchase channel,3,64000
4,Mens history,2,64000
5,Tenure,2,64000
6,Womens history,2,64000


Only pre-treatment attributes appear. Slicing outcomes by something measured
*after* the send would condition on a post-treatment variable and break the
comparison between arms — the same trap as the conditional conversion rate above.

### Visit lift by segment

`v_segment_metrics` gives every (dimension, level, arm) cell its outcome rates and
its lift over the control customers *within that same cell*.

In [7]:
recency = query("""
    SELECT level, arm, customers, min_arm_customers, visit_rate, visit_lift
    FROM v_segment_metrics
    WHERE dimension = 'Recency'
    ORDER BY level, arm
""")
recency.style.format({
    "customers": "{:,}",
    "min_arm_customers": "{:,}",
    "visit_rate": "{:.2%}",
    "visit_lift": "{:+.2%}",
}).hide(axis="index")

level,arm,customers,min_arm_customers,visit_rate,visit_lift
1-3 months,Mens E-Mail,"7,469","7,438",22.28%,+8.50%
1-3 months,Womens E-Mail,"7,486","7,438",18.18%,+4.40%
10-12 months,Mens E-Mail,"4,477","4,403",14.94%,+7.04%
10-12 months,Womens E-Mail,"4,521","4,403",12.25%,+4.35%
4-6 months,Mens E-Mail,"4,673","4,673",18.13%,+7.55%
4-6 months,Womens E-Mail,"4,753","4,753",14.62%,+4.05%
7-9 months,Mens E-Mail,"4,688","4,688",15.23%,+7.04%
7-9 months,Womens E-Mail,"4,627","4,627",13.57%,+5.38%


Visit lift looks fairly stable across recency bands for Mens E-Mail, which is a hint
that recency alone may not be where the interesting heterogeneity lives.

`min_arm_customers` is carried on every row for a reason: a lift computed from a few
hundred customers per side is mostly noise. Surfacing the number that governs the
uncertainty is more honest than silently dropping small cells — and it is a reminder
that scanning this table for the biggest lift is precisely how to fool yourself.
Feature 7 does subgroup analysis properly, with a multiple-comparison correction and
a distinction between pre-registered and exploratory slices.

## 6. Cross-checking SQL against pandas

Every metric in these views is recomputed in pandas by the test suite and asserted
equal. A view that is merely self-consistent can still be wrong; agreement between
two independent implementations is what catches a mis-specified GROUP BY or a
denominator taken over the wrong set.

In [8]:
from src.data.load import load_processed
from src import config

df = load_processed()
sql_rates = table("v_arm_metrics").set_index("arm")["visit_rate"]
pandas_rates = df.groupby(config.TREATMENT_COL, observed=True)["visit"].mean()

comparison = pd.DataFrame({
    "sql": sql_rates,
    "pandas": pandas_rates,
})
comparison["max_abs_diff"] = (comparison["sql"] - comparison["pandas"]).abs()
comparison.style.format({"sql": "{:.10f}", "pandas": "{:.10f}", "max_abs_diff": "{:.2e}"})

,sql,pandas,max_abs_diff
No E-Mail,0.1061672768,0.1061672768,0.00e+00
Mens E-Mail,0.1827568405,0.1827568405,0.00e+00
Womens E-Mail,0.1514003834,0.1514003834,0.00e+00


## Summary

| View | Answers |
|---|---|
| `v_arm_metrics` | What did each arm do? |
| `v_arm_lift` | How much better than control, per outcome? |
| `v_funnel` | Where in the funnel does the effect sit? |
| `v_customer_dimensions` | Long-format customer attributes for slicing |
| `v_segment_metrics` | Outcome rates and lift per (dimension, level, arm) |

Everything here is **descriptive**. Two things stand out for later features:

- Spend has a standard deviation ~14x its mean, which will dominate the power analysis.
- Both email arms show large relative lifts, but nothing has been tested yet.

**Next — Feature 3:** before trusting any of these differences, check that the
randomisation actually worked — sample ratio mismatch and covariate balance.